# Semana 3 - Feature Engineering

**Carga horaria:** 10h  
**Entregavel:** Dataset com features transformadas

## Objetivos
- Criar novas features a partir das variaveis originais.
- Aplicar transformacoes para melhorar representacao dos dados.
- Exportar dataset pronto para treino de modelos.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [ ]:
input_file = Path('data/iris_limpo.csv')
if not input_file.exists():
    raise FileNotFoundError('Execute primeiro o notebook da Semana 2 para gerar data/iris_limpo.csv')

df = pd.read_csv(input_file)
print('Shape base:', df.shape)
display(df.head())

In [ ]:
feature_df = df.copy()

# Features geometricas
feature_df['sepal_area'] = feature_df['sepal length (cm)'] * feature_df['sepal width (cm)']
feature_df['petal_area'] = feature_df['petal length (cm)'] * feature_df['petal width (cm)']

# Razao entre petala e sepala
feature_df['petal_to_sepal_ratio'] = feature_df['petal_area'] / (feature_df['sepal_area'] + 1e-9)

# Classe de tamanho da petala por quantil
feature_df['petal_size_class'] = pd.qcut(feature_df['petal_area'], q=3, labels=['small', 'medium', 'large'])

display(feature_df.head())

In [ ]:
# Escalonar colunas numericas derivadas
scale_cols = ['sepal_area', 'petal_area', 'petal_to_sepal_ratio']
scaler = StandardScaler()
feature_df[[c + '_z' for c in scale_cols]] = scaler.fit_transform(feature_df[scale_cols])

# One-hot da classe de tamanho
feature_df = pd.get_dummies(feature_df, columns=['petal_size_class'], drop_first=False)

print('Shape com features:', feature_df.shape)
display(feature_df.head())

In [ ]:
# Validacao do entregavel
required = {
    'sepal_area', 'petal_area', 'petal_to_sepal_ratio',
    'sepal_area_z', 'petal_area_z', 'petal_to_sepal_ratio_z'
}
assert required.issubset(feature_df.columns), 'Features obrigatorias ausentes.'

dummy_cols = [c for c in feature_df.columns if c.startswith('petal_size_class_')]
assert len(dummy_cols) >= 3, 'One-hot de petal_size_class incompleto.'

print('Entregavel Semana 3 OK: dataset com features transformadas.')

In [ ]:
output = Path('data/iris_features.csv')
feature_df.to_csv(output, index=False)
print('Arquivo salvo em', output)